<a href="https://colab.research.google.com/github/TapasviNomula/Air-Aware-smart-Air-Quality-prediction-system/blob/main/Ticket_Categorizer_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()

Saving synthetic_ticket_dataset_varied_1000.xlsx to synthetic_ticket_dataset_varied_1000.xlsx


In [3]:
import pandas as pd

df = pd.read_excel("synthetic_ticket_dataset_varied_1000.xlsx")

df.head()

,text,category
0,"Dear Support Team,\nsubscription renewed autom...",Billing
1,"Good morning, charged twice for my subscriptio...",Billing
2,"Good evening, invoice for last month is missin...",Billing
3,"Hi Team, requesting a refund for cancelled ord...",Billing
4,"Hi Team, refund is still pending.",Billing


In [4]:
df.to_csv("dataset.csv", index=False)

In [5]:
df = pd.read_csv("dataset.csv")

In [6]:
print(df.shape)
print(df.columns)
print(df['category'].value_counts())
df.head()

(1000, 2)
Index(['text', 'category'], dtype='object')
category
Billing      250
Technical    250
HR           250
General      250
Name: count, dtype: int64


,text,category
0,"Dear Support Team,\nsubscription renewed autom...",Billing
1,"Good morning, charged twice for my subscriptio...",Billing
2,"Good evening, invoice for last month is missin...",Billing
3,"Hi Team, requesting a refund for cancelled ord...",Billing
4,"Hi Team, refund is still pending.",Billing


In [7]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [10]:
df['clean_text'] = df['text'].apply(clean_text)

df[['text', 'clean_text']].head()

,text,clean_text
0,"Dear Support Team,\nsubscription renewed autom...",dear support team subscription renewed automat...
1,"Good morning, charged twice for my subscriptio...",good morning charged twice subscription regard
2,"Good evening, invoice for last month is missin...",good evening invoice last month missing thanks...
3,"Hi Team, requesting a refund for cancelled ord...",hi team requesting refund cancelled order awai...
4,"Hi Team, refund is still pending.",hi team refund still pending


In [11]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", len(X_train))
print("Testing :", len(X_test))

Training: 800
Testing : 200


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(800, 854)
(200, 854)


In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [14]:
y_pred = model.predict(X_test_tfidf)

In [15]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Accuracy: 1.0
Accuracy (%): 100.0


In [16]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     Billing       1.00      1.00      1.00        50
     General       1.00      1.00      1.00        50
          HR       1.00      1.00      1.00        50
   Technical       1.00      1.00      1.00        50

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [18]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[50  0  0  0]
 [ 0 50  0  0]
 [ 0  0 50  0]
 [ 0  0  0 50]]


In [21]:
new_tickets = [
    "My payment was deducted twice but I did not receive confirmation.",
    "The application crashes every time I try to log in.",
    "Please send my salary slip for this month.",
    "Can you tell me about your pricing plans?",
    "The server is down and returning a 500 Internal Server Error."
]

cleaned = [clean_text(ticket) for ticket in new_tickets]

vectors = tfidf.transform(cleaned)

predictions = model.predict(vectors)

for ticket, category in zip(new_tickets, predictions):
    print(f"Ticket: {ticket}")
    print(f"Predicted Category: {category}")
    print(" "*60)

Ticket: My payment was deducted twice but I did not receive confirmation.
Predicted Category: Billing
                                                            
Ticket: The application crashes every time I try to log in.
Predicted Category: General
                                                            
Ticket: Please send my salary slip for this month.
Predicted Category: HR
                                                            
Ticket: Can you tell me about your pricing plans?
Predicted Category: General
                                                            
Ticket: The server is down and returning a 500 Internal Server Error.
Predicted Category: Technical
                                                            


In [23]:
probabilities = model.predict_proba(vectors)

for ticket, pred, prob in zip(new_tickets, predictions, probabilities):
    confidence = max(prob) * 100
    print(f"Ticket: {ticket}")
    print(f"Predicted Category: {pred}")
    print(f"Confidence: {confidence:.2f}%")
    print(" " * 60)

Ticket: My payment was deducted twice but I did not receive confirmation.
Predicted Category: Billing
Confidence: 84.44%
                                                            
Ticket: The application crashes every time I try to log in.
Predicted Category: General
Confidence: 39.63%
                                                            
Ticket: Please send my salary slip for this month.
Predicted Category: HR
Confidence: 75.49%
                                                            
Ticket: Can you tell me about your pricing plans?
Predicted Category: General
Confidence: 83.07%
                                                            
Ticket: The server is down and returning a 500 Internal Server Error.
Predicted Category: Technical
Confidence: 85.55%
                                                            


In [24]:
for ticket, pred, prob in zip(new_tickets, predictions, probabilities):

    confidence = max(prob) * 100

    if confidence < 60:
        status = "Needs Human Review"
    else:
        status = "Auto Assigned"

    print("Ticket:", ticket)
    print("Category:", pred)
    print(f"Confidence: {confidence:.2f}%")
    print("Status:", status)
    print(" " * 60)

Ticket: My payment was deducted twice but I did not receive confirmation.
Category: Billing
Confidence: 84.44%
Status: Auto Assigned
                                                            
Ticket: The application crashes every time I try to log in.
Category: General
Confidence: 39.63%
Status: Needs Human Review
                                                            
Ticket: Please send my salary slip for this month.
Category: HR
Confidence: 75.49%
Status: Auto Assigned
                                                            
Ticket: Can you tell me about your pricing plans?
Category: General
Confidence: 83.07%
Status: Auto Assigned
                                                            
Ticket: The server is down and returning a 500 Internal Server Error.
Category: Technical
Confidence: 85.55%
Status: Auto Assigned
                                                            


In [25]:
urgent_keywords = [
    "urgent",
    "immediately",
    "critical",
    "server down",
    "500",
    "error",
    "crash",
    "failed",
    "payment failed"
]

def detect_priority(ticket):
    ticket = ticket.lower()

    for word in urgent_keywords:
        if word in ticket:
            return "High"

    return "Normal"

for ticket in new_tickets:
    print(ticket)
    print("Priority:", detect_priority(ticket))
    print(" " * 60)

My payment was deducted twice but I did not receive confirmation.
Priority: Normal
                                                            
The application crashes every time I try to log in.
Priority: High
                                                            
Please send my salary slip for this month.
Priority: Normal
                                                            
Can you tell me about your pricing plans?
Priority: Normal
                                                            
The server is down and returning a 500 Internal Server Error.
Priority: High
                                                            


In [26]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(tfidf, "tfidf.pkl")

print("Model saved successfully!")

Model saved successfully!
